# Advanced RAG Techniques

A working reference on retrieval-augmented generation: chunking, encoders, query rewriting, query expansion, re-ranking, GraphRAG, and semantic chunking with LLMs - with the Claude-specific production levers that matter when Claude is the generation model.

**Table of contents**

- [Part I - Chunking, Encoders, Query Rewriting](#part-i--chunking-encoders-query-rewriting)
    - [1. Chunking](#1-chunking--deciding-what-a-retrievable-unit-is)
    - [2. Encoders](#2-encoders--how-text-becomes-something-you-can-search)
    - [3. Query Rewriting](#3-query-rewriting--fixing-the-query-before-it-ever-hits-the-index)
    - [Claude-side production levers](#claude-side-production-levers)
- [Part II - Query Expansion, Re-ranking & GraphRAG](#part-ii--query-expansion-re-ranking--graphrag)
    - [1. Query Expansion](#1-query-expansion)
    - [2. Re-ranking](#2-re-ranking)
    - [3. GraphRAG](#3-graphrag)
    - [How they compose](#how-they-compose)
    - [Quick-reference: Claude API facts used above](#quick-reference-claude-api-facts-used-above)
- [Part III - Semantic Chunking with LLMs](#semantic-chunking-with-llms)
    - [1. What semantic chunking is](#1-what-semantic-chunking-is)
    - [2. How it helps you build advanced RAG *without* LangChain](#2-how-semantic-chunking-helps-you-build-advanced-rag-without-langchain)
    - [3. Pros and cons of building RAG without LangChain](#3-pros-and-cons--building-rag-without-langchain-using-semantic-chunking)
    - [4. One important caveat before you adopt it](#4-one-important-caveat-before-you-adopt-semantic-chunking)
---

# Part I – Chunking, Encoders, Query Rewriting

Retrieval quality, not generation quality, is where most RAG systems fail. All three of these techniques attack the same underlying mismatch: **the unit you store is not the unit the user asks about.**

---

## 1. Chunking – deciding what a "retrievable unit" is

**Fixed-size + overlap** (e.g. 512 tokens, 15% overlap) is the baseline. It's cheap and it never fails, but it slices mid-sentence and mid-table, and the overlap is a blunt hedge against boundary loss.

**Recursive character splitting** improves on this by trying separators in priority order – paragraph break, then newline, then sentence, then word – falling back only when a chunk still exceeds the limit. This is the sane default for prose.

**Structure-aware chunking** uses the document's own boundaries instead of guessing: Markdown headings, HTML DOM, function/class boundaries in code, rows-plus-header for tables. For code and technical docs this beats every size-based method, because the author already marked the semantic units for you.

**Semantic chunking** embeds sentences, then cuts where consecutive-sentence similarity drops below a threshold – boundaries land where the topic actually shifts. It costs an embedding pass

over the corpus at index time and is sensitive to the threshold, so treat it as a tuning option rather than a default.

Three structural patterns matter more than the splitter choice:

- **Small-to-big / parent-document retrieval** - index small, precise chunks (a sentence or two) for matching, but return the enclosing parent section to the model. You get the precision of small vectors and the context of large passages.
- **Contextual chunk headers** - prepend document title and heading path to each chunk before embedding ("Employee Handbook > Leave Policy > Parental Leave: ..."). Fixes the classic failure where a chunk says "eligible employees receive 12 weeks" with no indication of *what* or *whose* policy. A stronger variant sends each chunk plus the whole document to an LLM once at index time and stores a generated one-line situating summary alongside it - expensive, but this is exactly what the Batches API is for (50% cost reduction, up to 100,000 requests per batch; key results by `custom_id`, never by position, since they come back in arbitrary order).
- **Late chunking** - embed the full document with a long-context encoder, then pool token embeddings into chunk vectors afterward, so each chunk vector carries whole-document context.

One concrete correctness note on sizing: measure chunks with Claude's own tokenizer via `messages.count_tokens`, not `tiktoken`. `tiktoken` is OpenAI's BPE and undercounts Claude tokens by roughly 15-20% on ordinary prose, worse on code and non-English text. If you size chunks and context budgets with it, your "1000-token" chunks are meaningfully larger than you think.

---

## 2. Encoders – how text becomes something you can search

- **Bi-encoders** embed query and document *independently* into one vector each; retrieval is an
approximate nearest-neighbor search. Because documents are embedded offline, this scales to millions of chunks – but query and document never interact, so the model can't reason about their relationship. Practical dials: embedding dimension (Matryoshka-trained models let you truncate a 1536-d vector to 256-d and keep most of the quality, cutting index size and search latency), and asymmetric prefixes (`"query: "` / `"passage: "`) that many models require to hit their benchmarked numbers. Also check whether the model was trained for symmetric similarity or for question->passage retrieval – they are different objectives and swapping them silently degrades recall.

**Cross-encoders** take query and document *together* as one input and output a relevance score. Full attention across the pair makes them substantially more accurate, and quadratically too expensive to run over the corpus. Hence the standard two-stage architecture: bi-encoder retrieves top ~50-100 candidates, cross-encoder reranks to the top ~5 that actually go in the prompt. Reranking is usually the single highest-ROI addition to a mediocre RAG pipeline – it fixes precision without touching your index.

You can also use Claude itself as the reranker – one Messages call that scores or orders the candidates, with `output_config.format` set to a JSON schema so you get a clean ordered list rather than prose you have to parse. Slower and pricier than a dedicated reranker, but it handles instruction-style relevance ("prefer sources from after 2024") that a trained cross-encoder can't.

**Sparse retrieval** (BM25) is not obsolete. Lexical matching wins on exact identifiers, error codes, product SKUs, rare proper nouns – precisely the terms dense encoders blur together. This is why Anthropic ships BM25 as a first-class tool-search primitive (`tool_search_tool_bm25_20251119`) rather than treating everything as embeddings.

**Hybrid search** runs both and fuses. Reciprocal rank fusion is the usual choice because it combines *ranks* rather than scores, so you don't have to normalize incomparable scales: each


document scores `Σ 1/(k + rank_i)` across retrievers, with `k ≈ 60`. Dense catches paraphrase, sparse catches exact tokens, RRF needs no tuning.

Worth being explicit: **Anthropic does not expose an embeddings endpoint.** Everything goes through `POST /v1/messages`. So the encoder and reranker in a Claude-based RAG stack come from elsewhere - a dedicated embedding provider, or an open-weights model you host - while Claude does the query rewriting, reranking-by-instruction, and synthesis. Anyone who tells you to call an Anthropic embeddings API is working from a false memory.

---

## ## 3. Query rewriting - fixing the query before it ever hits the index

The user's literal question is often the worst possible search string. Rewriting is a cheap pre-retrieval step that pays for itself.

**Query expansion** adds synonyms, expands acronyms, and normalizes jargon, so "PTO carryover" also matches "paid time off rollover."

**Multi-query fan-out** generates 3-5 paraphrases from different angles, retrieves for each, and fuses the result sets with RRF. This is the most reliable recall fix available: it hedges against the single-embedding lottery where one unlucky phrasing misses the right chunk entirely.

**HyDE** (Hypothetical Document Embeddings) inverts the problem. Instead of embedding the question, have the model *write a fake answer* and embed that. A hypothetical passage lives in the same vector neighborhood as real passages, so you're comparing document-to-document instead of question-to-document - closing the asymmetry gap. It works surprisingly well on domains the encoder wasn't trained on, and it fails when the model hallucinates a passage about the wrong

topic.

**Step-back prompting** goes the other direction: abstract the specific question into a broader one ("What was the tax treatment of Section 174 R&D expenses in FY2023?" -> "How are R&D expenses treated for tax purposes?"), retrieve the general principle, then answer the specific case with both in context.

**Decomposition** splits multi-hop questions into independently answerable sub-questions. "Did the CFO who signed the 2023 filing also work at the acquirer?" is unanswerable by any single chunk; as two sub-queries it's trivial. Sub-queries can run in parallel when independent, or sequentially when one's answer feeds the next.

**Conversational condensation** is the unglamorous one that breaks the most production systems. "What about for contractors?" is a meaningless search string; rewriting it against the dialogue history into "What is the parental leave policy for contractors?" is the whole fix. If your chatbot's retrieval degrades after turn two, this is almost always why.

Tier the cost: rewriting is a small, well-specified transformation, so run it on `claude-haiku-4-5` or on `claude-opus-5` at `output_config: {effort: "low"}`, and save full-effort Opus 5 for synthesis. Constrain the output with a JSON schema (`output_config.format`, or `client.messages.parse()` with a Pydantic/Zod model) so you get a list of strings, not a numbered list wrapped in an apology.

---

## ## Claude-side production levers

**Ground the answer with Citations.** Set `citations: {enabled: true}` on each `document` content
block and Claude returns text split into blocks where cited spans carry a `citations` array with `cited_text`, `document_index`, `document_title`, and a location - `char_location` (`start_char_index`/`end_char_index`), `page_location` (1-indexed page numbers), or `content_block_location`. This is enforced grounding rather than a "please cite your sources" instruction, and it gives you the spans needed to render clickable provenance. It's GA, no beta header. One incompatibility to plan around: **citations and `output_config.format` cannot be used together** - you'll get a 400. If you need both a cited answer and structured fields, that's two calls.

**Get the cache breakpoint in the right place.** Prompts render as `tools` -> `system` -> `messages`, and caching is strict prefix matching - any byte change invalidates everything after it. RAG is the canonical shared-prefix / varying-suffix shape: stable corpus preamble, varying question. Put `cache_control` at the end of the *shared* portion, not at the end of the whole prompt. Get this backwards and every request writes a distinct cache entry that is never read - you pay the 1.25x write premium forever and get zero reads. Minimum cacheable prefix is model-dependent and *not* monotonic with capability: 512 tokens on Opus 5 and Fable 5, 1024 on Sonnet 5 and Opus 4.8, 4096 on Opus 4.5/4.6 and Haiku 4.5. Reads cost ~0.1x, so a 5-minute TTL breaks even at two requests. When fanning out multi-query retrieval over a shared prefix, send one request first and wait for its first streamed token before firing the rest - otherwise the concurrent requests all miss and all write.

**Index offline with Batches.** Contextual-header generation, chunk summarization, and synthetic-question generation are all embarrassingly parallel and latency-insensitive: half price, and results retained 29 days.

---

# Part II - Query Expansion, Re-ranking & GraphRAG
These three sit at different points in the pipeline – expansion widens the net *before* retrieval, re-ranking tightens it *after*, and GraphRAG changes the data structure being retrieved from at all.

---

## 1. Query Expansion

The premise: a single query embedding is one point in a high-dimensional space, and whether it lands near the right chunk is partly luck of phrasing. Expansion trades a little compute for many shots at the target.

### Lexical expansion (no LLM)

The classical technique is **pseudo-relevance feedback** (Rocchio, RM3). Retrieve the top-k documents, assume they're relevant, extract their high-IDF terms, and add those terms to the query with a weight – then retrieve again. Cheap, deterministic, and effective on BM25.

Its failure mode is **query drift**: if the initial top-k is off-topic, you expand *toward* the wrong topic and the second retrieval is worse than the first. Guard against it by keeping the original query heavily weighted (typically 0.6–0.8) and only pulling terms from documents above a score floor.

### Generative expansion (LLM-driven)

Four distinct shapes, and they are not interchangeable:
**Paraphrase fan-out.** Generate 3-5 rewrites at different levels of specificity and vocabulary, retrieve for each, fuse. This is the workhorse. The critical detail is that the paraphrases must be *diverse* - if you prompt for "rewrite this query 5 ways" you often get five near-identical strings and gain nothing. Prompt for varied angles explicitly: one more formal, one using domain jargon, one as a keyword-only string, one from the perspective of a document that would answer it.

**Sub-query decomposition.** For conjunctive or multi-hop questions, expansion means *splitting*, not paraphrasing. "Which of our SOC2 controls changed after the 2024 audit?" is two retrievals joined by set logic, not one fuzzy search.

**HyDE.** Generate a hypothetical answer passage and embed that instead of the question. This is expansion in embedding space rather than term space, and it specifically closes the question->passage asymmetry gap.

**Synthetic-question indexing** - the inverted variant, and often the best of the four. Instead of expanding at query time, expand at *index* time: for each chunk, generate 3-5 questions that chunk answers, and index those questions alongside (or instead of) the chunk text. Now query-to-index matching is question-to-question, which is symmetric and much easier for a bi-encoder. It moves all the cost offline - perfect Batches API work at 50% off - and adds zero query latency. The tradeoff is that your index only answers questions you anticipated, so keep the raw chunk vector too and search both.

### Fusing the results

Never concatenate and hope. Use **reciprocal rank fusion**: each document scores `Σ 1/(k + rank_i)` summed across the N retrievers it appears in, `k ≈ 60`. RRF operates on ranks, so cosine similarities and BM25 scores fuse without normalization, and documents found by *multiple* expansions get boosted - which is exactly the signal you want, since agreement across


paraphrases is evidence of genuine relevance.

### When to skip it

Expansion increases recall and *decreases* precision. If you already have a strong re-ranker downstream, expansion is nearly free upside – the re-ranker cleans up the noise. Without a re-ranker, aggressive expansion actively hurts, because you've stuffed the context window with plausible-but-irrelevant chunks. **Expansion and re-ranking are a matched pair; deploying expansion alone is usually a regression.**

---

## 2. Re-ranking

Re-ranking exists because first-stage retrieval optimizes for *recall at 100* while your prompt needs *precision at 5*. Those are different problems and want different models.

### The architectural spectrum

| Stage | Model | Query-doc interaction | Cost | Corpus scale |
| :--- | :--- | :--- | :--- | :--- |
| First-stage dense | Bi-encoder | None (independent vectors) | Offline index, ~1ms search | Millions |
| First-stage sparse | BM25 | Term overlap only | Inverted index | Millions |
| Late interaction | ColBERT | Per-token max-sim | Large index (vector per token) | ~100Ks |
| Re-rank | Cross-encoder | Full cross-attention | ~10-50ms per pair | 50-200 candidates |
| Re-rank | LLM | Full reasoning | ~1s per batch | 10-50 candidates |

**ColBERT / late interaction** is the interesting middle ground: it stores one vector per token and
scores by summing, for each query token, its maximum similarity against any document token. You get much of the cross-encoder's token-level precision while keeping documents pre-computed – but the index is 10-100x larger, which is why it's less common in production than either neighbor.

**Cross-encoders** concatenate `[query, document]` and run full attention over the pair, emitting a single relevance logit. Every query token can attend to every document token, so the model can resolve negation, qualifiers, and conditionals that a bi-encoder flattens into one averaged vector. Cost is `O(candidates)` forward passes per query, which caps you around 50–200 candidates in interactive latency budgets.

### LLM-as-re-ranker

Three prompting strategies, in increasing quality and cost:

- **Pointwise** – score each document independently, 0-10. Parallelizable, but scores across separate calls aren't calibrated against each other.
- **Pairwise** – ask "is A or B more relevant?" and sort. High quality, but `O(n log n)` calls.
- **Listwise** – hand the model all candidates at once and ask for a ranked ordering. One call, and the model sees candidates *in comparison*, which is where its judgment is best. Constrain the output with `output_config: {format: {type: "json_schema", ...}}` so you get an ordered array of IDs rather than prose. This is the practical default.

Listwise breaks down when candidates exceed the context you want to pay for; the standard fix is a **sliding window** – rank a window of 20, slide by 10, keeping the survivors – which approximates a global ordering in linear calls.

An LLM re-ranker's real advantage isn't raw accuracy over a good cross-encoder; it's that it accepts *instructions*. "Prefer primary sources." "deprioritize anything superseded." "the user is
an admin so include internal runbooks" are relevance criteria you cannot express to a trained cross-encoder. For cost, tier it: `claude-haiku-4-5`, or `claude-opus-5` at `output_config: {effort: "low"}`, and reserve full effort for synthesis.

### Beyond relevance ordering

Two refinements that matter more than people expect:

**Diversity / MMR.** Top-5 by pure relevance is frequently five near-duplicate chunks from the same document. Maximal Marginal Relevance penalizes candidates similar to those already selected, so the final set *covers* the question rather than restating one facet of it. This is critical when the answer requires synthesis across sources.

**Score thresholding - the abstention path.** Re-rankers produce calibrated absolute scores, not just an ordering. Use them: if the top score is below a floor, return *nothing* and let the model say it doesn't know. A pipeline that always returns its best 5 chunks will confidently answer questions the corpus doesn't cover, which is the most damaging failure mode in production RAG.

**Contextual compression.** After re-ranking, run an extraction pass that strips each surviving chunk to only the sentences bearing on the query. Cuts tokens substantially and reduces the lost-in-the-middle effect. Note that compression rewrites the text, which breaks character-offset provenance – so if you rely on Claude's Citations (`cited_text` with `char_location`), either compress by *whole sentences* and remap offsets, or skip compression on documents you need cited.

---

## 3. GraphRAG
Vector RAG retrieves *passages that resemble the query*. That framing has a hard ceiling: it cannot answer questions whose answer exists in no single passage.

Two question classes break it:

1. **Multi-hop relational** – "Which of our vendors are subsidiaries of companies we've had a security incident with?" The vendor list is in one document, the ownership structure in another, incidents in a third. No chunk resembles the query, and no chunk contains the answer.
2. **Global / corpus-level** – "What are the main themes across these 400 incident reports?" The answer is a property of the whole corpus. Top-k retrieval returns 5 reports, so the answer is structurally unreachable – you'd have to be lucky enough that some chunk already *stated* the summary.

GraphRAG addresses both by building a queryable knowledge structure at index time instead of a flat vector store.

### Indexing pipeline

**Step 1 - Extract entities and relationships.** Run an LLM over each chunk to pull out typed entities (person, org, system, policy) and typed relationships between them, each with a supporting description and a pointer back to the source chunk. Provenance is mandatory, not optional – it's what lets you cite the eventual answer.

**Step 2 - Resolve entities.** "Acme Corp," "ACME," and "Acme Corporation" must collapse to one node, or your graph fragments into disconnected islands and the whole approach fails. This is the step that most often sinks a GraphRAG project. Combine embedding similarity, string normalization, and an LLM adjudication pass on ambiguous clusters – and expect to iterate.
**Step 3 - Summarize elements.** An entity mentioned in 40 chunks accumulates 40 descriptions. Summarize them into one coherent node description, which becomes the node's searchable text.

**Step 4 - Detect communities.** Run hierarchical community detection (Leiden is the standard choice) to partition the graph into clusters of densely-connected entities, at multiple levels of granularity.

**Step 5 - Summarize communities, bottom-up.** Generate a report for each leaf community, then summarize those into parent-community reports, up to the root. You now have a **hierarchy of pre-computed summaries at every level of abstraction** - this is the artifact that makes global questions answerable, and it's why GraphRAG is expensive to build.

### Query modes

**Local search** - for entity-centric questions. Identify entities in the query, land on their nodes, traverse outward 1-2 hops, and assemble a context of the neighborhood: entity descriptions, relationships, and the source chunks behind them. This is what answers the multi-hop relational question - you *walk* to the answer rather than hoping one chunk contains it.

**Global search** - for corpus-level questions. Ignore the raw chunks entirely. Take community reports at an appropriate hierarchy level, map the question over each in parallel to produce partial answers with self-rated relevance scores, discard the low-rated ones, then reduce the survivors into a final answer. It's map-reduce over a pre-built summary tree, and it answers "what are the main themes" correctly because it genuinely surveys the whole corpus.

**DRIFT / hybrid search** - start global for orientation, then follow up with local traversals on the entities that surfaced. Usually the best quality, and the most calls.


### The honest tradeoffs

GraphRAG's cost is real and front-loaded. Indexing means an LLM pass over every chunk for extraction, plus entity resolution, plus a summarization pass per community per hierarchy level – easily 10-100x the indexing cost of a vector store. Global search is also expensive at query time, since it fans out over many community reports.

Both costs are addressable in ways worth knowing:

- Extraction and community summarization are offline, parallel, and latency-insensitive – **Batches API**, 50% off, up to 100,000 requests per batch. Key results by `custom_id`; they return in arbitrary order.
- Extraction wants a fixed schema, so use structured outputs (`output_config.format`, or `client.messages.parse()` with a Pydantic/Zod model). Note the schema constraints: `additionalProperties` must be `false`, no recursive schemas, and numeric/length bounds like `minimum` / `maxLength` aren't supported - validate those yourself after parsing.
- Global search's map step shares a large fixed prefix (the system prompt and instructions) across every community. Put the `cache_control` breakpoint at the end of the **shared** portion, not the end of the prompt – otherwise every map call writes a fresh cache entry at the 1.25x premium and reads none of them. Reads are ~0.1x, so this is the difference between global search being viable and not. On `claude-opus-5` the minimum cacheable prefix is 512 tokens; it's 1024 on Sonnet 5, and 4096 on Haiku 4.5 – non-obvious, so check before assuming. When fanning out, send one request and wait for its first streamed token before firing the rest, so the others hit a warm cache.
- Re-indexing is the real operational pain. Adding documents can merge previously-separate entities and reshape communities, so naive incremental updates drift out of sync with a full rebuild. Budget for periodic full re-indexing, or scope the graph to a slowly-changing corpus.

The other risk is that **extraction errors are silent and compounding**. A hallucinated relationship becomes a graph edge, gets summarized into a community report, and is then presented as fact with no vector-similarity check to contradict it. Vector RAG at least always returns real text. Keep source chunk provenance on every node and edge so answers stay auditable, and use Claude's Citations (`citations: {enabled: true}`) when you surface the underlying chunks – remembering that citations and `output_config.format` are mutually exclusive, so your extraction calls and your cited-answer calls are necessarily separate requests.

### When GraphRAG is the wrong tool

If your questions are "what does the docs say about X" – factoid lookups answerable by one passage – GraphRAG adds enormous cost for no gain, and plain hybrid retrieval plus a re-ranker wins on
every axis. GraphRAG earns its cost only when questions are genuinely relational or genuinely global.

A pragmatic middle path: keep hybrid vector retrieval as the default, add a lightweight entity index (entity -> chunks mentioning it) for multi-hop questions, and pre-compute summaries only at the document and section level rather than building a full community hierarchy. That captures a solid share of GraphRAG's benefit at a small fraction of its indexing cost – and it's a reasonable place to start before committing to the full pipeline.

---

## How they compose

The ordering constraint worth internalizing: **expansion without re-ranking degrades precision, and re-ranking without expansion can only reorder what a single query already found.** Add them together or the pipeline gets worse before it gets better.

---

## Quick-reference: Claude API facts used above

| Topic | Fact |
| :--- | :--- |
| Embeddings | **No Anthropic embeddings endpoint.** Everything goes through `POST /v1/messages`. Encoder/reranker come from another provider or self-hosted. |
| Token counting | `messages.count_tokens`  **not** `tiktoken`, which undercounts Claude tokens ~15-20% on prose, worse on code/non-English. |
| Grounding | `citations: {enabled: true}` per `document` block - `cited_text`, `document_index`, `document_title`, plus `char_location` / `page_location` / `content_block_location`. GA, no beta header. |
| Incompatibility | Citations **cannot** be combined with `output_config.format` (400). Use two calls. |
| Structured output | `output_config: {format: {type: "json_schema", schema: ...}}`, or `client.messages.parse()`. `additionalProperties` must be `false`; no recursive schemas; no `minimum`/`maximum`/`minLength`/`maxLength`. |


| Prompt caching | Sender order `tools` -> `system` -> `messages`; strict prefix match; max 4 breakpoints. Place `cache_control` at the end of the **shared** prefix. Reads ~0.1x; writes 1.25x (5m) / 2x (1h). |
| Cache minimums | 512 tokens (Opus 5, Fable 5) - 1024 (Opus 4.8, Sonnet 5, Sonnet 4.6) - 4096 (Opus 4.6, Opus 4.5, Haiku 4.5). Not monotonic - check per model. |
| Fan-out timing | Send one request, await its first streamed token, then fire the remaining N-1 so they hit a warm cache. |
| Batches | 50% cost reduction; 100,000 requests / 256 MB per batch; results retained 29 days. **Results arrive in any order - key by `custom_id`, never by position.** |
| Sparse retrieval | BM25 is available as a first-class tool-search primitive: `tool_search_tool_bm25_20251119`. |
| Effort tiering | `output_config: {effort: "low"|"medium"|"high"|"xhigh"|"max"}` (default "high"). Low effort on 'claude-haiku-4-5' for rewrite/rerank; full effort 'claude-opus-5' for synthesis. |
| Model default | `claude-opus-5` ($5/$25 per Mtok, 1M context). Never append date suffixes to these model IDs. |
| Thinking | `thinking: {type: "adaptive"}` for anything non-trivial. `budget_tokens` is rejected with a 400 on Opus 5 / 4.6 / 4.5, Fable 5, Sonnet 5.1 |

---

# # Semantic Chunking with LLMs

Part I introduced semantic chunking in one paragraph as a tuning option. This section is the full treatment: what it actually is, the three generations of the technique, and the specific question of whether it lets you build an advanced RAG pipeline **without LangChain** – with honest pros and cons rather than a recommendation dressed up as analysis.

---

## ## 1. What semantic chunking is

Every size-based splitter answers the question **"how long should a chunk be?"** Semantic chunking answers a different question: **"where does this document stop talking about one thing and start talking about another?"** Length becomes a constraint to respect, not the criterion to split on.

The motivation is a specific, common failure. A fixed-size splitter cuts at token 512 regardless of what's there, so a policy's condition lands in chunk 7 and its exception lands in chunk 8. Retrieve chunk 7 and the model confidently states a rule that the very next sentence overrides. Overlap mitigates this by accident; semantic boundaries prevent it by construction.

### ### Generation 1 – Embedding-based breakpoint detection

The original (and cheapest) approach uses no LLM at all, only an encoder:
1. Split the document into sentences.
2. Build a sliding window around each sentence – sentence *i* combined with its 'buffer' neighbours on either side – so each embedding reflects local context rather than a single sentence.
3. Embed every window.
4. Compute cosine **distance** between consecutive window embeddings. This is a 1-D signal of "how much did the topic just change."
5. Cut where the distance spikes above a threshold.

The threshold is the whole game, and there are four standard strategies:

| Strategy | Rule | Behaviour |
| :--- | :--- | :--- |
| **Percentile** | Cut where distance > 95th percentile | Predictable chunk count; the |
| **Standard deviation** | Cut where distance > mean + 3σ | Conservative; few, large chunks |
| **Interquartile** | Cut where distance > Q3 + 1.5·IQR | Outlier-driven; robust to noise |
| **Gradient** | Cut on the *derivative* of the distance curve | Better on documents with mixed layout |

Then clamp: enforce a `min_chunk_tokens` (merge runt chunks into their neighbour) and `max_chunk_tokens` (recursively split anything over the limit). Without clamps you get 5-token chunks that embed to noise and 6,000-token chunks that blow your context budget.
Cost is one embedding pass over the corpus at index time - real but modest, and entirely offline.

### Generation 2 - LLM boundary detection

Here the LLM replaces the distance heuristic and makes the judgment directly. It sees discourse structure an embedding distance cannot: "however," "in contrast," a shift from problem statement to proposed solution, the end of a worked example, the start of a new API's documentation.

There are four shapes, and the difference between them is mostly about **safety of the output**:

**(a) Index-emitting - the one to use.** Number the sentences, hand the model the numbered list, and ask it to return only the indices where a new topic begins, plus a short label per section. The model never reproduces the source text, so it *cannot* paraphrase, truncate, or drop content. Output is tiny (a JSON array of integers), so it's cheap and fast. Constrain it with `output_config: {format: ["type": "json_schema", ...])}` so you get `{"breakpoints": [1, 14, 31, ...]` rather than prose you have to parse.

**(b) Delimiter insertion - avoid.** Ask the model to return the document with `<|split|>` markers inserted. It reads well in a tutorial and fails in production; the model silently normalizes whitespace, fixes typos, or drops a paragraph, and now your chunks don't match your source. If you must use it, verify by stripping the delimiters and asserting byte-equality with the original - and expect that assertion to fire.

**(c) Sequential windowing (LumberChunker-style) - for long documents.** A whole book doesn't fit in one boundary-detection call, and quality degrades long before the context limit does. Instead: take a window of candidate segments, ask the model which segment starts a new topic, emit everything before it as a chunk, advance the window to that point, repeat. Cost is linear in document length, and each decision is made with tight local context.

**(d) Outline-first / hierarchical.** Ask the model for the document's logical outline, then map each outline entry back to a character offset. Produces a *tree* of chunks rather than a flat list, which pairs naturally with small-to-big retrieval - match on the leaf, return the section.

### Generation 3 - Propositional chunking

The most aggressive variant: decompose the text into **atomic, self-contained propositions**. Every


pronoun resolved, every implicit subject restored, each proposition true on its own.

> "It was ratified the following year." -> "The Nineteenth Amendment was ratified in 1920."

Retrieval precision goes way up, because each unit expresses exactly one fact with no ambiguity. Three real costs: your index grows several-fold; you lose all discourse structure and narrative flow; and the LLM is now *rewriting* your corpus, so factual drift in the propositions is a hallucination surface that no downstream check will catch. Use it as an \*additional\* index alongside the original chunks, retrieve on propositions but feed the model the parent passage - never as a wholesale replacement for your source text.

### The companion technique: contextual enrichment

Orthogonal to \*where\* you cut is \*what you store\*. For each chunk, make one LLM call with the chunk plus its parent document and generate a one-or-two-sentence situating preamble ("This section of the FY2024 10-K discusses segment revenue for the Cloud division..."), then prepend it before embedding. This is Anthropic's contextual retrieval pattern, and on most corpora it delivers a larger retrieval improvement than switching from fixed-size to semantic chunking does. **If you only have budget for one index-time LLM pass, spend it here, not on boundary detection.**

### Making it affordable

Boundary detection is offline, parallel, and latency-insensitive — the exact profile the **Batches API** exists for: 50% cost reduction, up to 100,000 requests per batch, results retained 29 days. Key results by `custom_id`, never by position, since they return in arbitrary order.

Beyond that: run it on `claude-haiku-4-5` or `claude-opus-5` at `output_config: {effort: "low"}` — boundary detection is a well-specified judgment, not a reasoning problem. If you make multiple passes over the same document (boundaries, then per-chunk context), put a `cache_control` breakpoint after the document so the second pass reads the cached prefix at ~0.1x instead of re-paying for it. Size chunks with `messages.count_tokens`, not `tiktoken` — see the quick-reference table above.

### Cache the boundaries, not just the prompt

LLM boundary detection is **non-deterministic**. Re-running your indexer must not silently reshuffle every chunk boundary, or your index churns, your embeddings all change, and any evaluation you ran
yesterday no longer describes today's system. Persist the computed breakpoints keyed by a hash of the source document, and only recompute when the document actually changes. Treat chunk boundaries as a versioned build artifact, not as something recomputed on every run.


## 2. How semantic chunking helps you build advanced RAG *without* LangChain

The connection isn't obvious at first, so here it is directly: **semantic chunking is the point in the pipeline where a framework helps you least and costs you most, and it's simultaneously the change with the largest effect on final answer quality.** That combination is what makes "no framework" a viable posture rather than a purity exercise.

Four concrete mechanisms:

### It's one prompt and one loop, not an orchestration problem

Frameworks earn their keep on *orchestration* - many components, many adapters, conditional routing. Boundary detection has none of that shape. It is: segment sentences -> one Messages call with a JSON schema -> assemble chunks by offset -> clamp sizes -> write to the store. That's a few dozen lines you can hold in your head, and wrapping it in a `TextSplitter` subclass adds indirection without adding capability. When you write it yourself, you can also do things the interface doesn't expose - different strategies per document type, a cheap embedding-distance pre-pass to skip LLM calls on obviously-uniform documents, domain rules like "never split a table from its header."

### Better chunks mean less machinery downstream

This is the leverage. Retrieval quality problems get compensated for downstream - more aggressive query expansion, a bigger candidate pool, heavier reranking, larger context windows. Every one of those compensations is a component, and components are what make people reach for a framework. Fix the index and the runtime path collapses to something genuinely small: embed the query -> hybrid search -> rerank -> one Messages call with citations. There is very little left for a framework to orchestrate. **You move complexity from runtime (where a framework helps) to index time (where it doesn't).**

### It preserves character offsets, which is what makes citations line up
This is the most underrated practical argument. Semantic chunking, done properly, is \*offset bookkeeping\*: every chunk knows its `start_char` and `end_char` in the original document. Keep that end-to-end and Claude's Citations return `char_location` spans that map exactly back to your source — real, clickable provenance.

Framework document abstractions routinely break this. Loaders normalize whitespace, splitters return text without offsets, "Transformers" rewrite content in place, and by the time text reaches the model there is no reliable mapping back to the source file. You then reconstruct provenance by fuzzy string matching, which fails on exactly the documents where it matters. Owning the pipeline means owning the offsets.

### You get new API capabilities on the day they ship

Frameworks abstract over providers, so a Claude-specific capability has to be modelled, wrapped, released, and adopted before you can use it — and prompt-caching placement, `effort` tiers, adaptive thinking, Batches for indexing, structured-output schemas, and citation locations are all things frameworks either lag on or actively obscure. Prompt caching is the sharpest case: a framework that rebuilds the prompt string on each call, or reorders `tools` / `system` / `messages`, silently breaks the prefix match. You still pay the 1.25x write premium on every request and never get a read, and nothing in the API response tells you the framework did this — you find it on the invoice.

### What "without LangChain" actually means in practice

Not "write everything from scratch." It means \*\*libraries, not architecture\*\*: use a PDF parser, a sentence segmenter, a vector store client, the Anthropic SDK, and a reranker as ordinary dependencies you call directly — while owning the control flow and the data model yourself. In particular it's entirely reasonable to import a framework's \*document loaders\* for the genuinely tedious formats and hand-write everything from chunking onward. That captures most of the ecosystem benefit with none of the abstraction cost.

### The shape of the pipeline you own

   segment      -> sentences with {start_char, end_char} [library: sentence segmenter]
   boundaries   -> LLM call, JSON schema -> [indices]    [Batches, haiku or opus@effort:low]
                   (cache by document hash - do not recompute)
   assemble     -> chunks; clamp min/max via count_tokens
   enrich       -> per-chunk situating context           [Batches - highest-ROI pass]
   embed        -> dense vectors (+ BM25 index, + entity index if graph)
   store        -> chunk text, offsets, doc id, title, context, vectors
   
QUERY TIME  (thin, ~one screen of code)
   condense/rewrite -> cheap call, effort:low
   retrieve     -> dense + BM25 -> RRF fuse
   rerank       -> cross-encoder, or LLM listwise; threshold -> abstain
   answer       -> Messages, citations enabled, cache_control after shared prefix


**Correct token accounting.** You call `messages.count_tokens` on the exact payload you're sending. No hidden prompt scaffolding inflating your counts, and no `tiktoken` undercounting Claude tokens by 15-20%.

**Day-one access to new API features.** `effort` tiers, adaptive thinking, structured outputs, context editing, tool search, mid-conversation system messages — you adopt them when they ship, not when a wrapper is released.

**Debuggability.** A bad answer is traceable in one pass: which chunks came back, what they scored, what the final prompt was. No callback-manager indirection, no reverse-engineering what a chain actually sent. This compounds — most RAG work *is* debugging retrieval.

**Smaller dependency surface.** Fewer transitive packages, less version churn, fewer breaking changes imposed on your schedule, a smaller supply-chain footprint.

**Tunable where it matters.** Per-document-type chunking strategy, custom clamps, domain rules, hybrid fusion weights, abstention thresholds — all trivially adjustable because they're your code, not configuration options someone else chose to expose.

### Cons

**You build the boring 80%.** Format parsing (PDF tables, scanned docs, .docx, HTML boilerplate stripping), OCR, encoding detection, deduplication, incremental re-indexing, retry with backoff, rate-limit handling, concurrency control, chunk-store schema and its migrations. None of it is interesting, all of it is required, and this is genuinely where a framework saves the most time.

**No ecosystem of connectors.** Every source — Confluence, S3, Notion, Postgres, SharePoint, Slack — is an integration you write and maintain. If you have one corpus in one format, this costs nothing. If you have fifteen heterogeneous sources, it's the dominant cost of the project and probably decides the question on its own.

**Semantic chunking has no defaults, so you own the tuning.** Threshold strategy, percentile value, buffer size, min/max clamps, prompt wording — all yours to tune, on your corpus, against an eval set you also have to build. Frameworks ship opinionated defaults that are *good enough* to get moving.
**Non-determinism you must manage.** LLM boundaries vary run to run. You need boundary caching keyed by document hash, and a versioning story for chunk sets, or your index quietly churns and your evaluations stop meaning anything.

**LLM chunking costs real money and time.** An LLM pass over the entire corpus, sometimes two (boundaries + enrichment). Batching halves it; forgetting to batch is a genuine budget surprise on a large corpus. Re-indexing multiplies it.

**You build observability and evaluation yourself.** No LangSmith-equivalent traces. Recall@k, faithfulness, groundedness, latency and cost dashboards, regression tests on a golden set — all in-house. Skip it and you're tuning blind, which is worse than any framework overhead.

**Reinvention risk.** The failure modes are specific and known: naive score-concatenation instead of RRF, the wrong tokenizer, no abstention path, no diversity in the final set, cache breakpoint in the wrong place. A framework encodes some of this hard-won knowledge as defaults. Building it yourself means you need to *know* it — this document is largely a list of the things you need to know.

**Onboarding and hiring.** "It's LangChain" is a shared vocabulary with tutorials and a large pool of people who recognize the shapes. "It's our custom pipeline" is documentation you must write and maintain, and every new engineer starts from zero.

### The tradeoff, stated plainly

| Your situation | Reasonable choice |
|---|---|
| One or few corpora, one format, quality and cost matter, long-lived system | **Hand-built.** Semantic chunking + hybrid + rerank + Claude with citations. |
| Prototype, demo, or validating whether RAG helps at all | **Framework.** Speed to first answer dominates; migrate later if it works. |
| Many heterogeneous sources, connector breadth is the hard part | **Framework loaders, hand-built core.** Import the parsers; own chunking, retrieval, and the model call. |
| Team already fluent in a framework, deadline-driven | **Framework**, but pin down prompt-cache placement and citation offsets early - those are what you'll otherwise lose. |
| Indexing-heavy, document centric, graph/hierarchy needs | Consider **LlamaIndex** over LangChain - indexing is its centre of gravity - or hand-build if provenance is critical. |
The middle path is underrated and usually correct: \*\*framework libraries, hand-built architecture.\*\* The expensive coupling isn't importing a PDF loader; it's letting a framework own your prompt assembly, your document objects, and your control flow.


## 4. One important caveat before you adopt semantic chunking

Semantic chunking is \*\*not reliably better than fixed-size chunking.\*\* Published comparisons are mixed, and on plenty of corpora a recursive splitter with sensible overlap plus a good reranker matches or beats LLM-driven boundaries — at a fraction of the index cost. The gains concentrate in specific conditions: long unstructured documents, heterogeneous content with real topic drift, absent or unreliable structural markup, and questions whose answers span condition-plus-exception pairs that a blind cut would separate.

Where semantic chunking most often loses: documents that already have good structure (use the headings — structure-aware chunking is cheaper \*and\* better), short documents, and any corpus where you haven't yet added a reranker. \*\*Add the reranker first.\*\* It's cheaper, it's more reliable, and it will tell you how much of your problem was actually chunking.

So sequence the work by cost-effectiveness, and stop when the numbers stop moving:

1. Build a golden eval set — 50–200 real questions with the passages that should be retrieved. Nothing below this line is measurable without it.
2. Add hybrid retrieval (dense + BM25, RRF fused).
3. Add a reranker with a score threshold and an abstention path.
4. Add contextual enrichment of chunks (one Batches pass).
5. \*Then\* try semantic chunking, and A/B it against your current splitter on the same eval set.
6. Consider GraphRAG only if your questions are genuinely relational or corpus-global.

Adopting semantic chunking at step 5 instead of step 1 is the difference between a measured improvement and an expensive act of faith.

> The pseudocode above is deliberately language-agnostic — no project language is established in
> this repository. Say which language you want (and whether embedding/reranking is a hosted provider
> or self-hosted) and the index-time pipeline can be written out properly against the real SDK.
> the Batches + structured-outputs boundary-detection pass is the piece with the most non-obvious
> detail.


# Visualizing Embeddings with t-SNE for Advanced RAG Development

Companion to [`RAG_readme.md`](/RAG_readme.md). That document covers what to build - chunking, encoders, query rewriting, expansion, re-ranking, GraphRAG, semantic chunking. This one covers how to *see* what you built, so you stop guessing which lever to pull.

The core argument: RAG failures are almost always **retrieval** failures, and retrieval happens in a 768-3072 dimensional space you cannot inspect. You get a ranked list of chunks and a similarity score, and neither tells you *why* the right chunk lost. t-SNE turns that opaque space into a picture you can reason about. It is a **hypothesis generator, not a measurement instrument** - that distinction runs through this entire document, and getting it wrong is how teams talk themselves into expensive mistakes.

**Table of contents**

- [1. What t-SNE actually does](#1-what-t-sne-actually-does)
- [2. Getting the setup right](#2-getting-the-setup-right-most-misleading-plots-are-setup-errors)
- [3. Eleven diagnostics that change what you build](#3-eleven-diagnostics-that-change-what-you-build)
- [4. The triage table - plot symptom to pipeline fix](#4-the-triage-table---plot-symptom-to-pipeline-fix)
- [5. How to not fool yourself](#5-how-to-not-fool-yourself)
- [6. t-SNE vs UMAP vs PCA](#6-t-sne-vs-umap-vs-pca---which-and-when)
- [7. Quantitative companions](#7-quantitative-companions---the-numbers-that-decide)
- [8. A practical workflow](#8-a-practical-workflow)
- [9. Using Claude alongside the plot](#9-using-claude-alongside-the-plot)
- [10. Limits - when visualization is the wrong tool](#10-limits---when-visualization-is-the-wrong-tool)

## 1. What t-SNE actually does

t-Distributed Stochastic Neighbor Embedding takes high-dimensional points and lays them out in 2-D so that **near stays near**. Mechanically:

1. In the original space, convert pairwise distances into conditional probabilities using a Gaussian kernel per point, with the kernel bandwidth tuned so each point has an effective number of neighbours equal to the **perplexity**.
2. In 2-D, define a similar probability distribution but with a **Student-t** kernel (heavy tails).
3. Move the 2-D points to minimize the KL divergence between the two distributions.

Two consequences matter more than the mechanism:

**It optimizes local neighbourhoods, and only those.** The heavy-tailed low-dimensional kernel is what lets tight clusters separate cleanly on the page - but it also means large distances are deliberately not preserved. This is why the number-one rule is: **cluster separation on a t-SNE plot is not proportional to semantic distance.** Two clusters far apart on screen may be adjacent in the real space, and vice versa.

**It has no out-of-sample transform.** t-SNE fits a specific set of points; there is no learned function to project a *new* point into an existing map. You cannot embed a fresh user query onto yesterday's plot. To overlay queries you must include them in the fit from the start, or use UMAP or PCA, which do expose a `transform()`. This trips up nearly everyone building a live monitoring view, and it is the single strongest reason to reach for UMAP instead when the visualization needs to be recurring rather than exploratory.

Three things the plot never tells you, no matter how convincing it looks:

- **Distances between clusters** - an artifact of the optimization, not a fact about your data.
- **Cluster sizes / densities** - t-SNE expands sparse regions and compresses dense ones by design.
- **Empty space** - whitespace between blobs means nothing.


## 2. Getting the setup right (most misleading plots are setup errors)

**Use cosine distance, or L2-normalize first.** Retrieval embeddings are compared by cosine similarity. If you run t-SNE with default Euclidean distance on un-normalized vectors, you are visualizing a geometry your retriever does not use - vector *magnitude* (which usually tracks chunk length) will structure the plot, and you'll conclude something about topics when you're really looking at document lengths. Either set the metric to cosine or normalize the vectors to unit length beforehand. **This is the most common way these plots lie.**

**PCA-reduce to ~50 dimensions first.** Standard practice, and it does two useful things: strips
low-variance noise dimensions that distort neighbourhoods, and cuts runtime substantially. Also initialize the layout with PCA rather than randomly - it makes results far more stable and preserves a little more global structure.

**Sweep the perplexity.** Perplexity is roughly "how many neighbours count as local," typically 5-50. Low values fragment the data into many small islands; high values merge real distinctions into blobs. Run 5, 30, and 50 at minimum. **Structure that survives all three is probably real;** structure that appears at one setting is probably an artifact. **Treat the sweep as mandatory, not optional.**

**Fix the random seed.** t-SNE is stochastic. Without a fixed seed you cannot compare two runs, and "the clusters moved" will be indistinguishable from "the data changed."

**Subsample, but stratify.** Barnes-Hut t-SNE is roughly O(n log n) and gets slow past ~50-100k points. Sample down, but sample *proportionally across documents and sources* - a naive random sample of a corpus dominated by one huge document tells you about that document only.

For reference, the parameterization in scikit-learn (t-SNE tooling is effectively Python-only; `openTSNE` is faster for large corpora):

```python
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize

X = normalize(vectors)                   # cosine geometry - do not skip
X = PCA(n_components=50, random_state=0).fit_transform(X)
xy = TSNE(
    n_components=2,
    metric="cosine",
    perplexity=30,                       # sweep 5 / 30 / 50
    init="pca",
    learning_rate="auto",
    random_state=0                       # reproducibility
).fit_transform(X)
```




**Make it interactive.** A static scatter plot of anonymous dots is nearly useless. The value comes from hovering a point and seeing the chunk text, its source document, its heading path, and its character offsets. Build it with a library that supports hover tooltips and store the chunk id alongside each coordinate so you can jump from a suspicious dot back to the actual source text. Every diagnostic below depends on being able to do that.


## # 3. Eleven diagnostics that change what you build

This is the substance - the specific things you look for, and the decision each one drives.

### 3.1 Chunking quality

Plot all chunks, coloured by source document (or by section). What you're looking for:

- **Chunks from the same section clustering together** - healthy. Your splitter is respecting topic boundaries.
- **One document's chunks scattered uniformly across the whole map** - the splitter is cutting mid-topic, or the document genuinely covers many unrelated subjects and should be split into logical sub-documents first.
- **A dense, undifferentiated blob at the centre** - the classic **over-long chunk** signature. Long chunks average many topics into one vector, and averaged vectors collapse toward the centroid. Every long chunk becomes weakly similar to every query, which is the worst possible retrieval behaviour: nothing is ever strongly wrong, and nothing is ever strongly right. Fix by reducing chunk size (measured with Claude's `count_tokens`, not `tiktoken`).
- **Fine, stringy filaments** - chunks too small; you're embedding fragments that carry no standalone meaning.

This is also the cleanest way to A/B a splitter. Render the same corpus chunked two ways - recursive fixed-size versus semantic - side by side, coloured by section. Better-separated, section-coherent clusters is genuine evidence, and unlike a retrieval metric it tells you *how* it improved.

A sharper variant: pick one document, colour its chunks by **sequential index** on a gradient.
Coherent chunking traces a path — consecutive chunks land near each other, with visible jumps at real topic shifts. Random colour scatter means your boundaries have no relationship to the document's semantics.

### 3.2 Duplicate and boilerplate detection

Look for very tight, very dense micro-clusters. Hover them. They are almost always boilerplate: navigation bars, page headers and footers, copyright blocks, license text, "This page intentionally left blank," email signatures, repeated disclaimers.

This matters more than it sounds. Boilerplate chunks are near-identical, so they occupy a dense region of the space and get retrieved for a wide range of queries, crowding out real content in your top-k. On corpora built from scraped HTML or PDF exports this is frequently *the* dominant retrieval problem, and it is invisible in aggregate metrics while being unmistakable on a t-SNE plot. The fix is index-time deduplication and boilerplate stripping — cheap, and often the single largest quality win available.

### 3.3 The query-document asymmetry gap

Embed a set of real user queries alongside your chunks and plot both with distinct markers. (Because t-SNE has no out-of-sample transform, include the queries in the fit.)

- **Queries interspersed among relevant chunks** — your encoder handles the asymmetry well.
- **Queries forming their own separate cluster, away from all chunks** — the asymmetry problem, made visible. Short interrogative text and long declarative passages occupy different regions, so nearest-neighbour search among *chunks* is being driven by which chunks happen to sit nearest the query region rather than by relevance.

This is the most valuable single diagnostic in the list, because it directly justifies three specific fixes that are otherwise adopted on faith: **HyDE** (embed a hypothetical answer instead of the question, so you compare passage-to-passage), **synthetic-question indexing** (index generated questions per chunk, so you compare question-to-question), and switching to an encoder trained for asymmetric retrieval with its `query: ` / `passage: ` prefixes applied correctly. If your queries and chunks are already well mixed, HyDE will probably buy you nothing — and knowing that saves you an entire implementation.
### 3.4 Retrieval failure triage

The workhorse. For each failed query in your eval set, plot the query, the top-k that *were* retrieved, and the gold chunk that *should* have been. Where the gold chunk sits tells you which component is broken:

| Gold chunk position | Diagnosis | Fix |
|---|---|---|
| Nearby, just outside top-k | First-stage recall is fine; ranking is the problem | Add or improve the **re-ranker**; widen the candidate pool |
| In the right neighbourhood but the query landed elsewhere | Query phrasing mismatch | **Query rewriting / expansion**, HyDE |
| Far from the query, in a coherent cluster of related content | Embedding or chunking problem | Better encoder, hybrid BM25, re-chunk |
| Split across two distant points | The answer was **cut in half** by chunking | Larger chunks, or small-to-big / parent-document retrieval |
| Isolated with no neighbours | Rare vocabulary the dense encoder can't place | **Hybrid retrieval** — this is exactly BM25's job |

Without this, "retrieval is bad" is a single undifferentiated complaint and you fix it by trying things. With it, each failure names its own remedy. This is the highest-ROI use of the technique.

### 3.5 Encoder selection

Embed the same corpus with two or three candidate encoders and plot each, coloured by a label you already trust (product area, document type, ground-truth topic). The encoder whose map separates your labels most cleanly is the better fit *for your domain*, which is frequently not the one higher on a public leaderboard — general benchmarks say little about a corpus full of your internal part numbers and acronyms.

Confirm by eye, decide by number: compute a silhouette score on those labels and, ultimately, recall@k on your eval set. The plot narrows the field cheaply; the metric picks the winner.

### 3.6 Coverage and content-gap analysis

Plot your corpus, then overlay a sample of **real production queries**. Look for regions dense with queries but sparse in chunks. Those are questions your users actually ask that your corpus cannot answer.

No amount of retrieval tuning fixes a content gap — the answer isn't there. This diagnostic converts a RAG debugging session into a documentation backlog, and it's typically the finding non-engineering
stakeholders care about most.

### 3.7 Semantic chunking threshold tuning

When tuning the boundary threshold described in `RAG_readme.md` Part III, visualize the outcome rather than trusting a percentile. Too aggressive and you get fragment filaments; too conservative and you get the central blob. Plot chunk sets produced at several thresholds and pick the one with coherent, separated, section-aligned clusters — then verify on the eval set.

### 3.8 Re-ranker value validation

Plot the candidate pool for a query, coloured by first-stage rank, and mark what the re-ranker promoted into the final top-5. If the promoted items were already the nearest neighbours, your re-ranker is mostly agreeing with the encoder and adding little. If it consistently promotes items that sat *further out* in embedding space, it is contributing real signal — full cross-attention seeing something the bi-encoder's averaged vector could not. That's the difference between a re-ranker worth its latency and one worth removing.

### 3.9 GraphRAG entity resolution and community validation

Two high-value uses on the graph side:

**Entity resolution.** Plot entity nodes by their description embeddings. Duplicate entities — "Acme Corp" / "ACME" / "Acme Corporation" — appear as separate points at very small distances. Scanning for suspiciously tight pairs is an effective way to catch the fragmentation that, per `RAG_readme.md`, is the failure that most often sinks a GraphRAG project. You can generate merge candidates directly from this.

**Community coherence.** Colour nodes by their detected community (*Leiden*). Coherent communities look coherent. A community whose members are scattered across the map means the graph structure and the semantics disagree — usually a sign of extraction noise or bad resolution upstream, and a reason to distrust the community summaries that global search depends on.

### 3.10 Drift monitoring
Plot query embeddings by time bucket. When users start asking about things in regions your corpus doesn't cover — a new product launch, an incident, a seasonal shift — it shows up as a migrating cloud before it shows up as a support complaint. **Use UMAP for this**, not t-SNE; you need `transform()` to project new queries into a stable existing map rather than refitting and getting a differently-oriented picture every time.

### 3.11 Multi-tenant and access-control sanity check

If your index is shared across tenants or permission tiers, colour by tenant. Heavy overlap between tenants means that if a metadata filter is ever missing or misapplied, cross-tenant retrieval is semantically plausible and will return convincing wrong-tenant content. This doesn't replace enforcing filters at the query layer — it tells you how bad the blast radius is when enforcement fails, which is worth knowing before it does.


## ## 4. The triage table - plot symptom to pipeline fix

| What you see | Likely cause | Action |
|---|---|---|
| Dense central blob | Chunks too long; vectors averaged toward centroid | Reduce chunk size (`count_tokens`), structure-aware splitting |
| Stringy filaments, tiny clusters | Chunks too small to be meaningful | Raise `min_chunk_tokens`; merge runts |
| Tight micro-cluster of near-identical points | Boilerplate / duplicates | Index-time dedupe and boilerplate stripping |
| Queries in their own separate cluster | Query-passage asymmetry | HyDE, synthetic-question indexing, asymmetric encoder + prefixes |
| Gold chunk near query but outside top-k | Ranking, not recall | Add re-ranker; widen candidate pool |
| Gold chunk isolated, no neighbours | Rare tokens dense search can't place | Hybrid retrieval with BM25 |
| Answer split across two distant points | Chunk boundary cut through the answer | Parent-document / small-to-big retrieval |
| Query-dense region with no chunks | Content gap | Write the missing content |
| One document's chunks scattered everywhere | Splitter ignoring topic structure | Semantic or structure-aware chunking |
| Sequential chunk colours randomly scattered | Boundaries unrelated to document semantics | Revisit chunking strategy entirely |
| Near-duplicate entity points in graph | Entity resolution failure | Merge pass before community detection |
| Graph community scattered across the map | Extraction noise; community summaries untrustworthy | Fix extraction/resolution before trusting global search |


## ## 5. How to not fool yourself

t-SNE produces compelling pictures from pure noise if you let it. Guard rails, in rough order of importance:

**Never measure anything off the plot.** No distances, no cluster sizes, no densities, no gaps. Use the picture to form a hypothesis, then test the hypothesis with recall@k on your eval set. Every diagnostic in section 3 ends in a change you can measure — if a finding can't be converted into a measurable change, treat it as decoration.

**Never tune a threshold by eye.** Choosing a semantic-chunking percentile or a rerank cutoff because the plot "looks cleaner" optimizes for 2-D aesthetics, which is not a proxy for retrieval quality. Use the plot to narrow the candidate range, the eval set to pick the value.

**Sweep perplexity and reject fragile structure.** Covered above and worth repeating: if a cluster appears at perplexity 5 and vanishes at 30, it is very likely an artifact of the algorithm rather than a property of your corpus.

**Beware apophenia.** Humans see clusters in random point clouds reliably. Sanity-check by running t-SNE on Gaussian noise of the same shape once — the plot will still show apparent structure, and seeing that yourself calibrates how much to trust the real one.

**Don't confuse "clusters cleanly" with "retrieves well."** They correlate, but a corpus can cluster beautifully by document type while failing every question a user asks, because the axis it separates on isn't the axis queries discriminate on.

**Remember the compression.** You're projecting 1536 dimensions into 2. The information loss is enormous and non-uniform. Any 2-D view is one lossy shadow of the real geometry; a second method (UMAP, PCA) casting a different shadow is a cheap and worthwhile cross-check.


## ## 6. t-SNE vs UMAP vs PCA - which and when

| **PCA** | **t-SNE** | **UMAP** |

| | **PCA** | **t-SNE** | **UMAP** |
|---|---|---|---|
| Type | Linear | Non-linear | Non-linear |
| Preserves | Global variance | Local neighbourhoods | Local, some global |
| Out-of-sample `transform()` | **Yes** | **No** | **Yes** |
| Deterministic | Yes | No (seeded only) | No (seeded only) |
| Speed | Very fast | Slow (~O(n log n)) | Fast |
| Scales to | Millions | ~50-100k | Millions |
| Interpretable axes | Yes | No | No |
| Best for | Pre-reduction; quick variance check | One-off exploratory diagnosis | Recurring views, monitoring, pipelines |

Practical guidance:

**PCA** — always use it as a 50-dim pre-reduction step before t-SNE. On its own it's a fast way to check "how many dimensions actually carry my corpus's variance," which tells you whether Matryoshka truncation to 256-d is likely safe.

**t-SNE** — the best cluster-revealing view for exploratory work, which is exactly what sections 3.1 through 3.9 are. Use it when you sit down to diagnose a problem.

**UMAP** — choose it for anything that runs more than once: drift dashboards, live query overlays, CI checks on index quality. `transform()` is the deciding feature; a stable map you can project new points into is worth more than marginally prettier clusters.

**PaCMAP / TriMAP** — newer methods with better global-structure preservation. Worth trying if cross-cluster relationships genuinely matter to your analysis; not necessary for standard triage.

Rule of thumb: **t-SNE to understand, UMAP to monitor, PCA to preprocess.**

---

## ## 7. Quantitative companions — the numbers that decide

The plot generates hypotheses; these confirm them. Never ship a change justified only by a picture.

**Trustworthiness / continuity** — how faithfully the 2-D layout preserves the original neighbourhoods. Low trustworthiness means the plot itself is unreliable and you should not be drawing conclusions from it at all. Check this first.

**Silhouette score** on labels you already trust — quantifies the cluster separation you *think* you see, and lets you compare encoders numerically.
**Recall@k and MRR** on your golden eval set - the only metrics that describe the thing you actually care about. Every plot-driven change gets validated here.

**Answer faithfulness / groundedness** - end-to-end. Retrieval improvements that don't move this weren't the bottleneck.

**Intra- vs inter-document chunk similarity** - a scalar version of the 3.1 diagnostic. Chunks within a section should be more similar to each other than to chunks from other sections; if not, your chunking is not capturing topics, and you can track that as a single number over time.

**Nearest-neighbour duplicate rate** - fraction of chunks with a neighbour above ~0.95 cosine. Catches the boilerplate problem from 3.2 without a plot, and belongs in CI.

Note the pattern: several visual diagnostics have scalar equivalents. Use the plot to *discover* the problem, then encode the scalar as a regression check so it can't come back silently.


## ## 8. A practical workflow

A concrete order of operations for a diagnostic session:

1. **Persist the vectors.** Store embeddings alongside chunk id, source document, heading path, character offsets, and chunk index. Without offsets and ids you can't get from a dot back to the text, and the whole exercise stalls. (Keeping offsets end-to-end is also what makes Claude's `char_location` citations line up - see `RAG_readme.md`.)
2. **Build the golden eval set first** - 50-200 real questions with the chunks that should be retrieved. Nothing below this is verifiable without it.
3. **Normalize, PCA to 50, t-SNE at perplexity 5 / 30 / 50, fixed seed.**
4. **Render interactively** with hover text showing chunk text, document, and offsets.
5. **Pass 1 - corpus health.** Colour by document. Look for the blob, the filaments, the boilerplate micro-clusters. Fix these first; they're cheap and they distort everything downstream.
6. **Pass 2 - query alignment.** Overlay real queries. Decide whether you have an asymmetry problem, and therefore whether HyDE / synthetic-question indexing is worth implementing.
7. **Pass 3 - failure triage.** For every failure in the eval set, plot query + retrieved + gold, and classify it against the table in 3.4. Tally the classes.
8. **Fix in tally order**, highest count first. Re-measure recall@k after each change.
9. **Encode the findings as scalar CI checks** (duplicate rate, intra/inter-document similarity, recall@k) so regressions surface without a human looking at a picture.
10. **Switch to UMAP** for whatever you decided to watch continuously.

Step 7 is where the value concentrates. A tally of failure classes turns "retrieval is bad" into a ranked, actionable work list — and it usually reveals that one class accounts for most failures, which is a much smaller problem than the one you thought you had.

---

## ## 9. Using Claude alongside the plot

The visualization is unlabelled by nature — you get shapes, not explanations. A few ways the LLM side closes that gap, using the API facts from `RAG_readme.md`:

**Auto-label the clusters.** Run a clustering algorithm on the embeddings (or on the 2-D coordinates), sample 5–10 chunks per cluster, and ask Claude for a short label and a one-line description per cluster. Constrain with `output_config: {format: {type: "json_schema", ...}}` so you get `{"clusters": [{"id": 3, "label": "...", "summary": "..."}]}` rather than prose. Suddenly your map has a legend, and the coverage gap analysis in 3.6 becomes something you can hand to a stakeholder. Cheap work — `claude-haiku-4-5` or `claude-opus-5` at `output_config: {effort: "low"}`.

**Explain individual failures.** For a triaged failure, give Claude the query, the retrieved chunks, and the gold chunk, and ask which of the failure classes in 3.4 it fits and why. Useful for processing a large failure set consistently, and it catches cases where the human eye reads the geometry wrong.

**Generate merge candidates for entity resolution.** Feed the suspiciously close entity pairs from 3.9 to Claude for a same-or-different adjudication with a confidence score. This is exactly the LLM adjudication pass that entity resolution needs, and the plot is what nominated the candidates.

**Batch it.** All of the above are offline, parallel, and latency-insensitive — the **Batches API** profile: 50% cost reduction, up to 100,000 requests per batch, results retained 29 days. Key by `custom_id`, never by position.

One constraint to keep straight: **Anthropic exposes no embeddings endpoint** — everything routes through `POST /v1/messages`. The vectors you're visualizing come from your own encoder, hosted elsewhere or self-hosted. Claude's role here is labelling, adjudicating, and explaining what the geometry got wrong.
projection surfaced, not producing the projection's input.


## ## 10. Limits - when visualization is the wrong tool

Be clear-eyed about the ceiling:

**It doesn't scale to inspection of everything.** You look at samples. Rare-but-important failure modes may never appear in your sample, and t-SNE is not a completeness check.

**It's exploratory, not a regression test.** You cannot diff two t-SNE plots meaningfully — the layout is stochastic and rotation/reflection-invariant. Scalar metrics are what go in CI.

**It says nothing about generation.** Hallucination, poor instruction-following, bad citation behaviour, prompt-caching misplacement — none of that is visible in embedding space. This is a retrieval diagnostic only.

**Well-separated clusters are not the objective.** Retrieval quality is. They correlate; they are not the same thing, and optimizing the picture is a real trap.

**If you don't have an eval set yet, build that instead.** A golden set of 100 question-chunk pairs will improve your system more than any amount of visualization, because it makes every subsequent decision measurable. Visualization tells you *why* a number is bad; you need the number first.

Used in that spirit — hypothesis generator upstream of a measurement — t-SNE is one of the cheapest ways to turn RAG debugging from guesswork into a ranked list of specific, testable fixes.

> The one code block above is scikit-learn, since t-SNE tooling is effectively Python-only. The pipeline steps are otherwise language-agnostic — no project language is established in this repository. Say which language you're using (and whether embedding/reranking is a hosted provider or self-hosted) and this can be written out as a working diagnostic script against the real SDKs.


# Multiprocessing

Multiprocessing in a production RAG system is worth real money at index time and is usually the **wrong tool** at query time. That asymmetry is the whole story, and getting it backwards is one of the more common architectural mistakes in RAG deployments — teams reach for a process pool to
parallelize API calls, hit rate limits, triple their memory footprint, and get no speedup, because the bottleneck was never the CPU.

The examples below are Python, since the GIL is what makes this a decision at all; the reasoning transfers to any runtime, but the specific hazards (fork-vs-spawn, pickling, BLAS oversubscription) are language-shaped.

**Contents**

1. [First question: is the work actually CPU-bound?](#1-first-question-is-the-work-actually-cpu-bound)
2. [Index time - where multiprocessing genuinely wins](#2-index-time---where-multiprocessing-genuinely-wins)
3. [Query time - processes *across* requests, not *within* one](#3-query-time---processes-across-requests-not-within-one)
4. [Multiprocessing with query expansion, stage by stage](#4-multiprocessing-with-query-expansion-stage-by-stage)
5. [The gotchas that break process pools](#5-the-gotchas-that-break-process-pools)
6. [Claude API interactions](#6-claude-api-interactions)
7. [Reference architecture](#7-reference-architecture)
8. [Sizing and benchmarking](#8-sizing-and-benchmarking)
9. [Decision table](#9-decision-table)

---

## ## 1. First question: is the work actually CPU-bound?

Multiprocessing solves exactly one problem: **CPU-bound work serialized by the GIL.** It solves it by paying for separate interpreters, separate memory, and inter-process serialization. If your bottleneck is waiting on a network socket, you pay all of that cost and get nothing — worse than nothing, because you've also multiplied your request rate against a rate limit.

Classify every stage honestly before choosing a concurrency model:

| Stage | Bound by | Right tool |
|---|---|---|
| PDF/docx/HTML parsing, OCR | **CPU** | Process pool |
| Sentence segmentation, regex, text cleaning | **CPU** (GIL-held) | Process pool |
| Local tokenization | **CPU** (GIL-held in pure Python; released in fast Rust tokenizers) | Process pool, or threads with a Rust tokenizer |
| Deduplication, MinHash/SimHash | **CPU** | Process pool |

| Local embedding inference | CPU/GPU **compute** | **Batching** in one process — not processes |
| Hosted embedding API | **Network I/O** | `asyncio` / threads |
| Local ANN search (FAISS, hnswlib) | CPU, but **releases the GIL** | Threads |
| Hosted vector DB query | **Network I/O** | `asyncio` |
| Pure-Python BM25 scoring | **CPU** (GIL-held) | Process pool |
| Local cross encoder rerank | CPU/GPU **compute** | Batching, or a separate inference service |
| **Any Claude API call** | **Network I/O** | **`asyncio` — never processes** |
| RRF fusion, sorting, thresholding | CPU, microseconds | Keep inline |
| t-SNE / UMAP projection (see above) | **CPU** | Process pool, or the library's own `n_jobs` |

Two rows deserve emphasis because they're where the mistakes cluster:

**Local model inference is not a multiprocessing problem, it's a batching problem.** Four processes each running a forward pass on one query is dramatically slower than one process running a forward pass on a batch of four — the matrix multiply is already parallel across cores, and four processes just fight each other for the same BLAS threads while holding four copies of the model in RAM. Batch first; only consider processes if you've already batched and are still CPU-starved.

**Every LLM and hosted-API call is I/O.** A process spends that time blocked on a socket. `asyncio` handles thousands of concurrent waits in one thread at near-zero cost. Using processes here is the single most common misapplication in RAG code.

---

## ## 2. Index time — where multiprocessing genuinely wins

Ingestion is the natural home for process pools: it's CPU-heavy, offline, embarrassingly parallel, and latency-insensitive. Parsing a few hundred thousand PDFs is hours of pure CPU, and near-linear scaling across cores is a real, large win.

**The unit of parallelism is the document**, not the chunk. One worker owns a document end-to-end — parse -> clean -> segment -> chunk -> count tokens -> emit records — so all the intermediate state stays inside one process and only compact results cross the boundary. Chunk-level parallelism means shipping text back and forth for negligible per-item work, and serialization eats the gain.

What belongs in the pool:


- **Format parsing** — PDF text extraction, table detection, OCR, `.docx`, HTML boilerplate stripping. Usually C extensions doing heavy work, sometimes releasing the GIL, always faster in parallel. Typically the single largest ingest cost.
- **Sentence segmentation and cleaning** — regex-heavy pure Python, fully GIL-bound, scales well.
- **Token counting for chunk-size clamps** — if you're using a *local* tokenizer. Note the caveat in §6: Claude's `count_tokens` is an API call, not local work.
- **Deduplication and boilerplate detection** — MinHash signatures per document in the pool, then a single process join to find collisions. This is the fix for the boilerplate micro-clusters §3.2 identified, and it parallelizes cleanly.
- **BM25 index construction** — per-shard term frequency counting in workers, merged at the end.
- **GraphRAG post-processing** — normalizing extracted entity names, computing description embeddings, blocking candidate pairs for entity resolution.
- **The t-SNE/UMAP projections from this document** — one process per parameter setting, so the whole perplexity sweep (§2) runs concurrently instead of serially.

What does *not* belong in the pool, even at index time: the **LLM passes** — semantic boundary detection, contextual chunk enrichment, propositional decomposition, graph extraction. Those are API calls. Use the Batches API (§6), not a process pool.

The clean split for an ingest job is therefore:

```text
process pool -> parse, clean, segment, tokenize, dedupe      (CPU, scales with cores)
Batches API  -> boundaries, contextual enrichment, extraction (I/O, 50% cheaper, no fan-out code)
single process-> merge, dedupe join, index write, offset bookkeeping
```

---

## ## 3. Query time — processes *across* requests, not *within* one

At query time your budget is a few hundred milliseconds, and process overhead is measured in tens to hundreds of milliseconds. Spawning a process per request — or per expanded query — is a latency regression before it does any work.
The multiprocessing that **does** belong in production serving is the one you probably already have and may not think of as multiprocessing: **the web server's worker processes.** Gunicorn/Uvicorn workers, sized to roughly the core count, each running an event loop. That's how you use all your cores for a RAG service. Parallelism lives *between* concurrent requests; within a single request, use `asyncio`.

If a single request genuinely needs CPU-bound work — a local cross-encoder, a pure-Python BM25 pass — do not create processes inline. Two correct options:

- **A pre-warmed pool** created at startup, with the model loaded once per worker in an initializer, and the async handler dispatching via `loop.run_in_executor`. Startup cost is paid once.
- **A separate inference service** (TEI, vLLM, Triton, or your own) that the API workers call over the network. This turns CPU work into I/O work from the caller's perspective, which is strictly easier to scale, and lets you put the model on a GPU host while the API layer stays cheap. **For local embedding and reranking models in production, this is the right answer** — it decouples memory footprint, batching policy, and scaling from your request handlers.

---

## ## 4. Multiprocessing with query expansion, stage by stage

Query expansion looks like a textbook parallelism case — one query becomes N, N independent retrievals, fuse the results — so it's the place people reach for multiprocessing most often. Walk the stages and the picture inverts: **almost none of query expansion is CPU-bound.**

Recall the pipeline from `RAG_readme.md`: rewrite/condense -> expand to N queries (+ HyDE) -> retrieve per query -> RRF fuse -> rerank -> answer.

**Stage A — generate the rewrites.** One Claude call producing N paraphrases under a JSON schema, or optionally a small fan-out (a paraphrase call plus a HyDE call plus a decomposition call). Pure network I/O. `asyncio.gather` over the calls. Processes here buy nothing and cost a fork.

**Stage B — embed the N expanded queries.**
- *Hosted embedding API:* I/O-bound. Better still, most providers accept a **batch of inputs in one request** — so N queries becomes one round trip, and the concurrency question disappears. Do this before considering any parallelism.
- *Local model:* compute-bound, and the answer is batching, not processes. N queries in one forward
pass. A process pool here would hold load N copies of the encoder and slow you down.

**Stage C — N vector searches.**
- *Hosted vector DB:* I/O-bound — `asyncio.gather`, with a semaphore to cap in-flight requests.
- *Local FAISS/hnswlib:* CPU-bound, but these libraries **release the GIL** in their search calls, so a **thread pool** gives you real parallelism with zero serialization and one shared copy of the index. A process pool would give each worker its own copy of a multi-gigabyte index — usually fatal. If you must use processes, share the index via `mmap` or POSIX shared memory rather than pickling it.

**Stage D — BM25 for the N queries.** This is the one stage where a process pool is genuinely defensible at query time. Pure-Python BM25 implementations hold the GIL throughout scoring, so threads don't help. A pre-warmed pool with the index memory-mapped (or inherited by `fork` on Linux) parallelizes N queries properly. Better long-term fix: move BM25 into a store that already handles it natively — Elasticsearch/OpenSearch, or Postgres full-text — at which point it becomes I/O again.

**Stage E — RRF fusion.** Merging a few hundred candidates by `\sum 1/(k + rank_i)`. Microseconds. Never parallelize this; the coordination costs more than the work.

**Stage F — rerank.**
- *LLM listwise reranking:* I/O — asyncio. If you shard candidates into sliding windows, `gather` the window calls.
- *Local cross-encoder:* batch all (query, candidate) pairs into one forward pass. This is the stage where batching matters most, since you may be scoring 100 pairs.

So the honest summary: **query expansion is an `asyncio` problem with one batching decision and at most one process-pool exception.** The reason it *feels* like a multiprocessing problem is that the fan-out is real — you genuinely do N things at once — but "concurrent" and "parallel" are different requirements, and expansion needs the former.

Where multiprocessing *does* legitimately serve query expansion:

- **Across concurrent user requests** — the server worker processes. This is the real answer, and it scales expansion throughput linearly with cores.
- **Offline expansion** — pre-computing expansions for a known query distribution (FAQ sets, common support questions) and caching them. That's an index-time job: process pool for the local work,
Batches for the LLM work, and at serving time an expansion becomes a cache hit costing nothing.
- **Evaluation sweeps** — running your golden eval set across many expansion configurations to pick parameters. Genuinely CPU-and-throughput bound in aggregate, perfect for a pool, and completely offline.

That last one is worth its own note: **the highest-value use of multiprocessing around query expansion is not serving it, it's evaluating it.** Sweeping expansion strategies (paraphrase count, HyDE on/off, fusion 'k') across a few hundred eval questions is thousands of independent retrievals. Parallelize that and your tuning loop goes from an overnight job to a coffee break — which changes how many configurations you're willing to test, which is what actually improves the system.

The shape of the query-time handler, for reference:

```python
import asyncio

SEM = asyncio.Semaphore(8)            # cap in-flight calls; protects rate limits

async def guarded(coro):
    async with SEM:
        return await coro

async def expanded_retrieve(question: str, k: int = 100):
    rewrites = await generate_rewrites(question)    # 1 Claude call, JSON schema
    queries = [question, *rewrites]                 # keep the original — always
    
    vectors = await embed_batch(queries)            # ONE batched request, not N
    
    dense = [guarded(vector_search(v, k)) for v in vectors]
    sparse = [guarded(bm25_search(q, k)) for q in queries]  # pool-backed if pure Python
    results = await asyncio.gather(*dense, *sparse, return_exceptions=True)
    
    good = [r for r in results if not isinstance(r, Exception)]
    if not good:
        raise RetrievalUnavailable
    return rrf_fuse(good, k=60)                     # cheap, inline
```
Two production details in that sketch. `return_exceptions=True` with a partial-result fallback means one slow or failing retriever degrades recall instead of failing the request — with expansion you have N redundant retrievals, so partial results are usually still good answers. And the semaphore is not optional: expansion multiplies your request rate by N, so it is the stage most likely to trip provider rate limits.

---

## ## 5. The gotchas that break process pools

These are the failures that turn a process pool from a speedup into an outage. Most are Windows-relevant, which matters here.

**`fork` vs `spawn`.** Linux defaults to `fork` (cheap, inherits memory). **Windows and modern macOS use `spawn`**: a fresh interpreter that re-imports your module, so any module-level code runs again in every worker. Without an `if __name__ == "__main__":` guard you get infinite process spawning. With `spawn`, models and indexes are *not* inherited — each worker reloads them, so a 2 GB model means 2 GB * workers and a long cold start. Load expensive state in a pool `initializer`, once per worker.

**Never fork after initializing CUDA or a thread pool.** A forked child inherits a broken copy of the CUDA context and any locks held at fork time, producing hangs that reproduce intermittently and resist debugging. Create the pool *before* touching GPU state, and load models inside the workers.

**Pickling cost.** Every argument and return value is serialized. Passing a 100k x 1024 float32 embedding matrix between processes is ~400 MB of copying, which will exceed the compute you saved. **Return ids and scores**, not text or vectors; keep bulk data in shared memory, `mmap`, or on disk (Parquet/Arrow) and pass paths.

**BLAS oversubcription.** NumPy/PyTorch each spawn threads equal to core count. Eight workers * eight BLAS threads on eight cores is 64 threads thrashing, and it is reliably *slower* than serial. Set `OMP_NUM_THREADS=1` (and `MKL_NUM_THREADS=1`) in workers doing array math. This one silently costs people most of their expected speedup.

**Memory multiplication.** Even with `fork`'s copy-on-write, CPython's reference counting writes to
object headers, which dirties pages and gradually un-shares memory. Large Python-object structures degrade toward one full copy per worker. Prefer NumPy arrays and `mmap`, which stay genuinely shared.

**Result ordering.** Pool results arrive out of order (and `imap_unordered` guarantees it). **Key results by document/chunk id, never by position** — the same discipline the Batches API forces with `custom_id`, and a bug that produces plausible-looking, silently mismatched data.

**Rate limiting across processes.** A token bucket in shared memory across N processes is genuinely hard to get right; a token bucket in one async event loop is a few lines. Since the rate-limited work is I/O anyway, this is another argument for keeping all API traffic in one async process.

**Error visibility and timeouts.** A worker that segfaults on a malformed PDF can take down the pool. Use `concurrent.futures` with per-task timeouts, catch exceptions per item, log the offending document id, and continue. Tune `chunksize` for small tasks — the default often means the dispatch overhead dominates.

**Debuggability.** Breakpoints, profilers, and stack traces all get harder across process boundaries. Keep a single-process code path available via a config flag; you will want it.

---

## ## 6. Claude API interactions

**Do not build a process pool to fan out Claude calls.** They're I/O; `asyncio` is correct.

**Batches is the managed alternative to writing your own fan-out.** For index-time LLM work — semantic boundary detection, contextual enrichment, propositional decomposition, graph extraction, cluster labelling from §9 — the Batches API gives 50% cost reduction, up to 100,000 requests per batch, and results retained 29 days, with no concurrency code, no retry logic, and no rate-limit management on your side. Hand-rolled parallel fan-out is more code, more fragile, and twice the price. **Results return in arbitrary order - key by `custom_id`.**

**Parallelism defeats prompt caching if you're careless.** Concurrent requests sharing a large prefix all miss the cache and all pay the 1.25x write premium, because none of them has finished writing the entry the others would read. The fix is timing: send one request, await its first streamed token, then fire the remaining N-1 against the now-warm cache. Coordinating that across processes needs IPC;


in one event loop it's an `await`. **This is a concrete, billable reason to keep LLM fan-out single-process.** It bites hardest exactly where fan-out is widest — GraphRAG global search mapping over community reports, and multi-query expansion sharing a system prompt.

**`count_tokens` is a network call, not a local tokenizer.** Calling it per chunk inside a process pool means thousands of API round trips and rate-limit pressure, not parallel CPU work. For chunk-size clamps during ingest, either batch the counting offline or clamp on a local approximation and verify with `count_tokens` at the boundaries that matter. (Still never `tiktoken` for a real count — it undercounts Claude tokens by 15-20%.)

**Streaming doesn't cross process boundaries cleanly.** If you stream the final answer to a user, that request belongs in the async web worker, not dispatched to a pool.

---

## ## 7. Reference architecture

Where each concurrency model belongs in a production RAG deployment:

```text
INGEST (offline, throughput-optimized)
ProcessPoolExecutor(max_workers=cores-1, initializer=load_state)
  one task per DOCUMENT: parse -> clean -> segment -> chunk -> dedupe
  OMP_NUM_THREADS=1 in workers; return ids/records, not blobs
Batches API: boundary detection, contextual enrichment, extraction
  50% off - key by custom_id - no fan-out code
single process: merge, dedupe join, embed via batched API, index

SERVE (online, latency-optimized)
N web-server worker PROCESSES (= cores)  <- the real multiprocessing
each running one asyncio event loop
  rewrite/expand -> asyncio.gather over Claude calls
  embed          -> ONE batched request
  retrieve       -> asyncio.gather (+ thread pool for local ANN)
  BM25           -> search engine, or pre-warmed pool if pure-Python
```
  fuse/rerank    -> inline RRF; LLM rerank via async
  answer         -> streamed, citations enabled, cache_control set
  local encoder/reranker -> SEPARATE inference service (TEI/vLLM/Triton)

EVAL & TUNING (offline, the underrated win)
ProcessPoolExecutor over (eval question * configuration) grid
  expansion strategies, fusion k, chunk sizes, thresholds
ProcessPoolExecutor over t-SNE/UMAP parameter sweeps (§2 above)
## ## 9. Decision table

| You want to parallelize | Bound by | Use | Not |
|---|---|---|---|
| Parsing 500k PDFs | CPU | Process pool, one doc per task | `asyncio` |
| Semantic boundary detection over a corpus | API I/O | **`Batches API`** | Process pool |
| Contextual chunk enrichment | API I/O | **`Batches API`** | Process pool |
| Embedding a corpus (hosted) | I/O | Batched requests + `asyncio` | Process pool |
| Embedding a corpus (local model) | Compute | Large batches; separate inference service | Process pool |
| N expanded queries -> N searches | I/O (hosted) | `asyncio.gather` + semaphore | Process pool |
| N expanded queries -> local FAISS | CPU, GIL released | Thread pool, shared index | Process pool (index copies) |
| N expanded queries -> pure-Python BM25 | CPU, GIL held | Pre-warmed pool, mmap'd index | Inline loop |
| Generating rewrites / HyDE | API I/O | `asyncio.gather` | Process pool |
| RRF fusion | Trivial | Inline | Anything |
| LLM listwise reranking | API I/O | `asyncio`, sliding windows | Process pool |
| Serving concurrent user requests | Mixed | Web-server worker **processes** | One process |
| Eval sweeps over configurations | CPU + throughput | Process pool | Serial loop |
| t-SNE perplexity sweep | CPU | Process pool, one per setting | Serial loop |

The failure mode to remember, because it's the one that looks most reasonable on a whiteboard: **spawning processes to parallelize Claude API calls during query expansion.** It adds fork latency to a latency-critical path, multiplies memory, breaks prompt caching, makes rate limiting nearly impossible to coordinate, and delivers no speedup over three lines of `asyncio.gather` — because the work was never CPU-bound in the first place.

> Python throughout this section, since the GIL is what makes the choice load-bearing. Say which language you're targeting (and whether embedding/reranking is hosted or self-hosted) and the ingest pool and async query handler can be written out properly against the real SDKs.
